In [2]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("../data/processed/train_FD001_processed.csv")
val = pd.read_csv("../data/processed/val_FD001_processed.csv")

with open("../data/processed/feature_cols.json") as f:
    feature_cols = json.load(f)

X_train, y_train = train[feature_cols], train["RUL_clipped"]
X_val, y_val = val[feature_cols], val["RUL_clipped"]

X_test = pd.read_csv("../data/processed/X_test_FD001.csv")
y_test = pd.read_csv("../data/processed/y_test_FD001.csv")

print(X_train.shape, X_val.shape)
print(X_test.shape, y_test.shape)
print(X_test.iloc[:2, :3])
print(y_test.head(3))

(16561, 45) (4070, 45)
(100, 45) (100, 1)
   sensor_2  sensor_3  sensor_4
0 -0.195953 -1.523214 -1.111959
1 -0.256180 -0.640748  0.217605
     0
0  112
1   98
2   69


In [3]:
# To Create the Baseline model 
y_test = y_test.iloc[:, 0]

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_pred - y_true))

def nasa_score(y_true, y_pred):
    d = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    return np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1))

def evaluate(y_true, y_pred):
    return {"RMSE": rmse(y_true, y_pred), "MAE": mae(y_true, y_pred),
            "NASA": nasa_score(y_true, y_pred)}

baseline_pred = np.full(len(y_val), y_train.mean())
print("y_test shape:", y_test.shape)
print("train mean RUL:", round(y_train.mean(), 1))
print(evaluate(y_val, baseline_pred))

y_test shape: (100,)
train mean RUL: 87.0
{'RMSE': np.float64(41.72082538935907), 'MAE': np.float64(36.929802663876686), 'NASA': np.float64(1285855.8101210925)}


In [4]:
# Linear Regression
from sklearn.linear_model import LinearRegression

results = {}
results["Baseline (mean)"] = evaluate(y_val, baseline_pred)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_val_pred = np.clip(lr_model.predict(X_val), 0, y_train.max())
results["Linear Regression"] = evaluate(y_val, lr_val_pred)

pd.DataFrame(results).T.round(2)

,RMSE,MAE,NASA
Baseline (mean),41.72,36.93,1285855.81
Linear Regression,18.03,14.07,21524.00


In [5]:
# Rigge
from sklearn.linear_model import Ridge

results = {}
results["Baseline (mean)"] = evaluate(y_val, baseline_pred)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

ridge_pred = np.clip(ridge.predict(X_val), 0, y_train.max())
results["Ridge"] = evaluate(y_val, ridge_pred)

pd.DataFrame(results).T.round(2)

,RMSE,MAE,NASA
Baseline (mean),41.72,36.93,1285855.81
Ridge,18.03,14.07,21522.96


In [6]:
# Random Forest 
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_val_pred = np.clip(rf.predict(X_val), 0, y_train.max())
results["Random Forest"] = evaluate(y_val, rf_val_pred)

pd.DataFrame(results).T.round(2)

,RMSE,MAE,NASA
Baseline (mean),41.72,36.93,1285855.81
Ridge,18.03,14.07,21522.96
Random Forest,16.81,11.75,31476.51


In [7]:
# Gradient Boosting Regressor

from sklearn.ensemble import GradientBoostingRegressor
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
# Train the model
gb_model.fit(X_train, y_train)

# Predict on validation data
gb_val_pred = np.clip(
    gb_model.predict(X_val),
    0,
    y_train.max()
)
# Evaluate the model
gb_results = evaluate(y_val, gb_val_pred)

print("Gradient Boosting Validation Results:")
print(gb_results)

Gradient Boosting Validation Results:
{'RMSE': np.float64(15.726553198911006), 'MAE': np.float64(11.801920743637288), 'NASA': np.float64(21603.46732744974)}


In [8]:
# Extra resgressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

missing_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=5,
        random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200, min_samples_leaf=5,
        random_state=42, n_jobs=-1),
}

for name, model in missing_models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_val), 0, y_train.max())
    results[name] = evaluate(y_val, pred)
    print("done:", name)

pd.DataFrame(results).T.round(2).sort_values("RMSE")


done: Linear Regression
done: Random Forest
done: Extra Trees


,RMSE,MAE,NASA
Extra Trees,15.52,11.12,21816.99
Random Forest,16.81,11.75,31476.51
Linear Regression,18.03,14.07,21524.00
Ridge,18.03,14.07,21522.96
Baseline (mean),41.72,36.93,1285855.81


In [9]:
# Gradient Bossting
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

missing_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=5,
        random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200, min_samples_leaf=5,
        random_state=42, n_jobs=-1),
}

for name, model in missing_models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_val), 0, y_train.max())
    results[name] = evaluate(y_val, pred)
    print("done:", name)

pd.DataFrame(results).T.round(2).sort_values("RMSE")


done: Linear Regression
done: Random Forest
done: Extra Trees


,RMSE,MAE,NASA
Extra Trees,15.52,11.12,21816.99
Random Forest,16.81,11.75,31476.51
Linear Regression,18.03,14.07,21524.00
Ridge,18.03,14.07,21522.96
Baseline (mean),41.72,36.93,1285855.81


In [10]:
# ## Final Test Evaluation (Gradient Boosting)
from sklearn.ensemble import HistGradientBoostingRegressor

all_df = pd.concat([train, val]).reset_index(drop=True)
X_all, y_all = all_df[feature_cols], all_df["RUL_clipped"]

final_model = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_depth=6, random_state=42
)
final_model.fit(X_all, y_all)

test_pred = np.clip(final_model.predict(X_test), 0, y_all.max())

print("Final test result:")
print(evaluate(y_test, test_pred))

comparison = pd.DataFrame({"true_RUL": y_test.values, "predicted_RUL": test_pred.round(1)})
comparison["error"] = (comparison["predicted_RUL"] - comparison["true_RUL"]).round(1)
comparison.head(10)

Final test result:
{'RMSE': np.float64(17.57453416404242), 'MAE': np.float64(12.81107498130789), 'NASA': np.float64(696.1616965651897)}


,true_RUL,predicted_RUL,error
0,112,125.0,13.0
1,98,117.1,19.1
2,69,48.1,-20.9
3,82,115.9,33.9
4,91,100.2,9.2
5,93,111.6,18.6
6,91,88.8,-2.2
7,95,95.8,0.8
8,111,103.2,-7.8
9,96,93.2,-2.8


In [11]:
# Comparasion all the model 

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

comparison_results = {}

# Baseline
baseline_pred = np.full(len(y_val), y_train.mean())
comparison_results["Baseline (Mean)"] = evaluate(
    y_val, baseline_pred
)

# Train and evaluate each model
for name, model in models.items():

    model.fit(X_train, y_train)

    val_pred = np.clip(
        model.predict(X_val),
        0,
        y_train.max()
    )

    comparison_results[name] = evaluate(y_val, val_pred)

    print(f"Completed: {name}")

# Display comparison
comparison_df = pd.DataFrame(comparison_results).T.round(2)

comparison_df.sort_values("RMSE")

Completed: Linear Regression
Completed: Ridge
Completed: Random Forest
Completed: Extra Trees
Completed: Gradient Boosting


,RMSE,MAE,NASA
Extra Trees,15.52,11.12,21816.99
Gradient Boosting,15.73,11.80,21603.47
Random Forest,16.81,11.75,31476.51
Linear Regression,18.03,14.07,21524.00
Ridge,18.03,14.07,21522.96
Baseline (Mean),41.72,36.93,1285855.81


In [12]:
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np
import pandas as pd

# Step 1: Combine training and validation data
all_df = pd.concat([train, val]).reset_index(drop=True)

X_all = all_df[feature_cols]
y_all = all_df["RUL_clipped"]

# Step 2: Train the final Extra Trees model
final_model = ExtraTreesRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_all, y_all)

# Step 3: Predict RUL for test engines
test_pred = np.clip(
    final_model.predict(X_test),
    0,
    y_all.max()
)

# Step 4: Evaluate the final model
print("Extra Trees - Final Test Results")
print(evaluate(y_test, test_pred))

# Step 5: Compare actual and predicted RUL
comparison = pd.DataFrame({
    "true_RUL": np.asarray(y_test).reshape(-1),
    "predicted_RUL": np.round(test_pred, 1)
})

comparison["error"] = (
    comparison["predicted_RUL"] - comparison["true_RUL"]
).round(1)

display(comparison.head(10))

Extra Trees - Final Test Results
{'RMSE': np.float64(17.866928807189236), 'MAE': np.float64(13.122389933261186), 'NASA': np.float64(747.631829011997)}


,true_RUL,predicted_RUL,error
0,112,122.9,10.9
1,98,119.2,21.2
2,69,61.8,-7.2
3,82,104.6,22.6
4,91,100.4,9.4
5,93,117.7,24.7
6,91,99.7,8.7
7,95,97.8,2.8
8,111,105.7,-5.3
9,96,98.9,2.9


In [ ]:
# R2 and adjusted r2
from sklearn.metrics import r2_score

def adjusted_r2(y_true, y_pred, n_features):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

p = len(feature_cols)

# Validation on Extra Trees 
et_val_model = ExtraTreesRegressor(
    n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1
).fit(X_train, y_train)
et_val_pred = np.clip(et_val_model.predict(X_val), 0, y_train.max())

print("Validation  R2:", round(r2_score(y_val, et_val_pred), 3),
      "| Adjusted R2:", round(adjusted_r2(y_val, et_val_pred, p), 3))

# Test  model 
print("Test        R2:", round(r2_score(y_test, test_pred), 3),
      "| Adjusted R2:", round(adjusted_r2(y_test, test_pred, p), 3))

Validation  R2: 0.862 | Adjusted R2: 0.86
Test        R2: 0.815 | Adjusted R2: 0.661


In [14]:
## Save the Final Model
import yaml
import joblib
import json

joblib.dump(final_model, "../models/rul_model_v1.joblib")

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

meta = {
    "dataset": cfg["data"]["dataset"],
    "model": "ExtraTreesRegressor",
    "window_size": cfg["features"]["window_size"],
    "rul_clip": cfg["features"]["rul_clip"],
    "n_features": len(feature_cols),
    "test_metrics": {
        k: float(v)
        for k, v in evaluate(y_test, test_pred).items()
    },
}

with open("../models/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(meta)


{'dataset': 'FD001', 'model': 'ExtraTreesRegressor', 'window_size': 30, 'rul_clip': 125, 'n_features': 45, 'test_metrics': {'RMSE': 17.866928807189236, 'MAE': 13.122389933261186, 'NASA': 747.631829011997}}


In [15]:
print("comparison_df:", "comparison_df" in globals())
print("y_test:", "y_test" in globals())
print("test_pred:", "test_pred" in globals())

comparison_df: True
y_test: True
test_pred: True


In [ ]:
# Model Comparasion
def df_to_md(df):
    cols = ["Model"] + list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for name, row in df.iterrows():
        lines.append("| " + " | ".join([str(name)] + [f"{v:,.2f}" for v in row]) + " |")
    return "\n".join(lines)

print(df_to_md(comparison_df.sort_values("RMSE")))


| Model | RMSE | MAE | NASA |
|---|---|---|---|
| Extra Trees | 15.52 | 11.12 | 21,816.99 |
| Gradient Boosting | 15.73 | 11.80 | 21,603.47 |
| Random Forest | 16.81 | 11.75 | 31,476.51 |
| Linear Regression | 18.03 | 14.07 | 21,524.00 |
| Ridge | 18.03 | 14.07 | 21,522.96 |
| Baseline (Mean) | 41.72 | 36.93 | 1,285,855.81 |


In [ ]:
# To make the report on this text 

from sklearn.metrics import r2_score

test_metrics = evaluate(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

report = f"""# Model Comparison - FD001

Validation = 20 held-out engines (split by engine, random_state=42).

{df_to_md(comparison_df.sort_values("RMSE"))}

**Selected model:** Extra Trees (lowest RMSE and MAE on validation).
The gap between the top two models is small.

## Official test set (100 engines, one prediction per engine)

| RMSE | MAE | R2 | NASA |
|---|---|---|---|
| {test_metrics['RMSE']:.2f} | {test_metrics['MAE']:.2f} | {test_r2:.3f} | {test_metrics['NASA']:.1f} |
"""

print(report)

# Model Comparison - FD001

Validation = 20 held-out engines (split by engine, random_state=42).

| Model | RMSE | MAE | NASA |
|---|---|---|---|
| Extra Trees | 15.52 | 11.12 | 21,816.99 |
| Gradient Boosting | 15.73 | 11.80 | 21,603.47 |
| Random Forest | 16.81 | 11.75 | 31,476.51 |
| Linear Regression | 18.03 | 14.07 | 21,524.00 |
| Ridge | 18.03 | 14.07 | 21,522.96 |
| Baseline (Mean) | 41.72 | 36.93 | 1,285,855.81 |

**Selected model:** Extra Trees (lowest RMSE and MAE on validation).
The gap between the top two models is small.

## Official test set (100 engines, one prediction per engine)

| RMSE | MAE | R2 | NASA |
|---|---|---|---|
| 17.87 | 13.12 | 0.815 | 747.6 |



In [20]:
# to Save this Comparasion into the reports

from pathlib import Path

Path("../reports/model_comparison.md").write_text(report, encoding="utf-8")

print("saved:", Path("../reports/model_comparison.md").exists())

saved: True
